# Run a baseline active-learning sampler

One of the 11 published baselines, one dataset, the full budget sweep. No
CellViT, no VLM, no text -- every baseline here selects on the frozen DINOv2
visual cache alone (`sampling.specs.BASELINE_SAMPLERS`). `scalpel`, this
project's own method, runs in `run_al_main.ipynb` instead.

Attach the Kaggle Dataset that `extract_visual_features.ipynb` published for
this (dataset, seed); this notebook does not extract features itself unless
that cache is missing, and extracting 90k images on the fly is a cost this
notebook is not meant to pay.

On a Kaggle **T4 x2** session the work is split across both GPUs, one worker
process per card. With several (seed, variant) jobs they are split job-wise;
with a single variant the BUDGET LIST is split instead (`SPLIT_BUDGETS`), which
is what keeps both cards busy on the common case of one sampler, one seed. A
prefix-exact sampler (`random`, `coreset`, `tcm`) derives its whole sweep from
one selection pass and so stays on one GPU by design. Set `PARALLEL = False`
to force serial.

Per budget this writes selected indices with their per-step acquisition trace
-- score, and wherever the method computes them separately, its uncertainty
and coverage terms -- plus the probe weights, the test predictions, the
metrics table, a sanity report and a run log. See `main.py`.

Metrics here are accuracy, precision, recall and macro-F1 only. PALM and any
other curve-level metric belong to `evaluate_al_sampler.ipynb`, which reads
these files.

The last cell packs everything into ONE zip at the top of `/kaggle/working`,
named `{dataset}_{sampler}_seed{seeds}`, and deletes the loose checkpoints --
the same shape both extraction notebooks use, because the Output tab is the
only way a file leaves a "Save & Run All" session.

Re-running the notebook skips variants whose `_results.pt` already exists, so a
session that hits the 12-hour limit can simply be run again.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/CryAndRRich/codapath.git"
REPO_BRANCH = "namhai"
REPO = Path("/kaggle/working/codapath")

if (REPO / ".git").is_dir():
    subprocess.check_call(["git", "-C", str(REPO), "fetch", "origin", REPO_BRANCH])
    subprocess.check_call(["git", "-C", str(REPO), "switch", REPO_BRANCH])
    subprocess.check_call(["git", "-C", str(REPO), "pull", "--ff-only", "origin", REPO_BRANCH])
elif REPO.exists():
    raise RuntimeError(f"{REPO} exists but is not a Git repository")
else:
    subprocess.check_call(
        ["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO)]
    )

branch = subprocess.check_output(
    ["git", "-C", str(REPO), "branch", "--show-current"], text=True
).strip()
assert branch == REPO_BRANCH, (branch, REPO_BRANCH)
print("repo:", REPO, "| branch:", branch)

In [ ]:
%cd /kaggle/working/codapath

In [ ]:
import os
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub", "hf-transfer"])

os.environ["TOKENIZERS_PARALLELISM"] = "false"
if "/kaggle/working/codapath" not in sys.path:
    sys.path.append("/kaggle/working/codapath")

In [ ]:
# ---- EDIT THIS CELL ----
DATASET = "pathmnist"
SEEDS = [42]          # several seeds run back to back; run_name keeps them apart

# ONE baseline per notebook run. `scalpel` is not in this list on purpose --
# see sampling.specs.BASELINE_SAMPLERS and run_al_main.ipynb.
#   random | coreset | typiclust | activeft | tcm
#   margin | entropy | badge | dropquery | uncertainty_herding | refine
SAMPLER = "uncertainty_herding"

# One dict per full budget sweep. `[{}]` runs config.yaml as-is, which is what
# every baseline wants by default. A few axes worth sweeping, from
# config/config.yaml:
#   typiclust : {"k_nn": 10}
#   tcm       : {"transition_class_multiple": 2}
#   activeft  : {"temperature": 0.1}
#   dropquery : {"dropout_ratio": 0.5}
# Sweeping a baseline's axes is baseline TUNING: either do it for every method
# or none, and say which in the report.
VARIANTS = [{}]

# Leave RUN_NAME None so each variant gets a collision-safe name of its own.
RUN_NAME = None

# Use both T4s when the session has them and there is more than one job.
PARALLEL = True

# Split ONE variant's budget list across both GPUs. This is what makes a
# single-variant run use both cards: without it there is one job, and one job
# can only occupy one GPU.
#
# It applies ONLY to a sampler that is not prefix-exact, where every budget is
# an independent run anyway (margin, entropy, badge, dropquery,
# uncertainty_herding, refine, typiclust, activeft). A prefix-exact sampler
# (random, coreset, tcm) derives its whole sweep from ONE selection pass, so
# sharding would repeat that pass per shard and cost more than it saves -- the
# next cell detects this and leaves it on one GPU. Nothing is lost by that:
# those three are also the cheap ones.
SPLIT_BUDGETS = True

# ---- TWO Kaggle Datasets, both required ----
#
# They are not interchangeable, and attaching only the feature one is the easy
# mistake to make: the feature cache holds the DINOv2 matrices and nothing
# else. The labels a sampler selects on, the labels the probe trains against,
# and the sample-order fingerprint that VALIDATES the cache itself all come
# from the raw images -- so `main.run` opens the dataset either way, and a
# missing DATA_ROOT fails before the cache is ever reached.
#
#   DATA_ROOT   raw images (pathmnist_224.npz, HistoSet, SkinTissue)
#   FEATURE_DIR the .npy features extract_visual_features.ipynb published,
#               which is what lets this notebook skip the backbone pass
#
# Both are starting points: the next cells search from here and print what
# they actually resolved to, so a remounted slug still works.
DATA_ROOT = "/kaggle/input/datasets/cryandrrich/nckh2026"
FEATURE_DIR = "/kaggle/input/datasets/nhtquyn/pathoactive"
OUTPUT_DIR = "/kaggle/working/checkpoints"

In [ ]:
from huggingface_hub import snapshot_download

print("Downloading facebook/dinov2-base ...")
snapshot_download(repo_id="facebook/dinov2-base")

In [ ]:
import yaml
import torch

import main
from sampling.specs import BASELINE_SAMPLERS, spec_for
from utils.parallel import run_variants_parallel, visible_gpu_count
from utils.kaggle import find_data_root, find_visual_cache
from utils.progress import format_duration

In [ ]:
# `find_data_root` searches the default Kaggle mount points too, so a
# DATA_ROOT that was remounted under a different slug is still found. It
# reports where it actually landed rather than assuming.
DATA_ROOT = find_data_root([Path(DATA_ROOT)])

DATA_PATHS = {
    "pathmnist": str(DATA_ROOT / "pathmnist_224.npz"),
    "histoset": str(DATA_ROOT / "HistoSet-5x14/HistoSet-5x14"),
    "skintissue": str(DATA_ROOT / "SkinTissue/SkinTissue/tiles"),
}
print("data root (raw images):", DATA_ROOT)

In [ ]:
with open("config/config.yaml", "r", encoding="utf-8") as handle:
    config = yaml.safe_load(handle)

# This notebook is baseline-only. A sampler that needs a CellViT cell view or a
# VLM text prior belongs in run_al_main.ipynb instead -- catching that here, in
# an assert message that names the right notebook, beats a mid-run failure deep
# inside main.py once one budget has already spent GPU time.
assert SAMPLER in BASELINE_SAMPLERS, (
    f"{SAMPLER!r} is not a baseline (BASELINE_SAMPLERS={sorted(BASELINE_SAMPLERS)}). "
    "scalpel and any encoder/text variant run in run_al_main.ipynb."
)
spec = spec_for(SAMPLER)
assert "cell_embeddings" not in spec.needs, f"{SAMPLER} needs a CellViT cache -- unexpected for a baseline"
assert "text_embeddings" not in spec.needs, f"{SAMPLER} needs a VLM text prior -- unexpected for a baseline"

dataset_info = config["datasets"][DATASET]
training_cfg = config["training"]
base_cfg = dict(config.get("samplers", {}).get(SAMPLER, {}))
sampler_cfgs = [{**base_cfg, **overrides} for overrides in VARIANTS]

data_path = Path(DATA_PATHS[DATASET])
assert data_path.exists(), f"Missing Kaggle input: {data_path}"
assert torch.cuda.is_available(), "Attach a Kaggle GPU before running AL"
assert VARIANTS, "VARIANTS needs at least one dict (use [{}])"
assert SEEDS, "SEEDS needs at least one seed"
assert RUN_NAME is None or len(VARIANTS) * len(SEEDS) == 1, (
    "A fixed RUN_NAME with several variants/seeds makes every run overwrite the previous one"
)

# The DINOv2 cache is optional: main.py re-extracts on a miss. It must not
# re-extract into a read-only /kaggle/input, which would only fail AFTER the
# whole forward pass, so fall back to a writable directory instead. Since the
# milestone this notebook exists for is "the cache is already published",
# missing it here means the wrong dataset was attached, not a routine event --
# print loudly rather than silently eating a 90k-image forward pass.
vit_name = config.get("models", {}).get("vit", "facebook/dinov2-base")
found = find_visual_cache(DATASET, SEEDS[0], vit_name, hint=FEATURE_DIR)
if found is not None:
    FEATURE_DIR = str(found)
    print("features cache:", FEATURE_DIR)
else:
    print(f"[features] WARNING: no cache found for {DATASET}/seed{SEEDS[0]}/{vit_name}")
    print(f"  under {FEATURE_DIR!r} or the default Kaggle input roots.")
    print("  Falling back to extracting it in THIS session -- check that the")
    print("  extract_visual_features.ipynb output dataset is attached if that")
    print("  was not the intent.")
    FEATURE_DIR = "/kaggle/working/features"

if not str(FEATURE_DIR).startswith("/kaggle/input"):
    Path(FEATURE_DIR).mkdir(parents=True, exist_ok=True)
SAVE_DIR = Path(OUTPUT_DIR) / DATASET
SAVE_DIR.mkdir(parents=True, exist_ok=True)

print(f"{SAMPLER}: {spec.passes} pass, prefix_exact={spec.prefix_exact} - {spec.why}")
print(f"GPUs visible: {visible_gpu_count()}")
for cfg in sampler_cfgs:
    print("  config:", cfg)

In [ ]:
import time

BUDGETS = config["cumulative_budget"]
workers = visible_gpu_count() if PARALLEL else 1

# Budget sharding only helps a sampler whose budgets are independent runs.
# `spec.prefix_exact` is exactly that property, so it -- not a hand-kept list
# of names -- decides, and a new sampler cannot drift out of sync with it.
shard_budgets = SPLIT_BUDGETS and workers > 1 and not spec.prefix_exact and len(BUDGETS) > 1
if SPLIT_BUDGETS and spec.prefix_exact:
    print(f"[shard] {SAMPLER} is prefix-exact: one shared selection pass covers every")
    print("        budget, so its sweep stays on a single GPU (sharding would repeat")
    print("        that pass per shard). This is expected, not a misconfiguration.")

def budget_shards(budgets, n):
    """Deal budgets round-robin so each shard gets a mix of cheap and expensive.

    Cost grows with the budget, so a contiguous split would hand one worker
    every large budget and leave the other idle for most of the session.
    """
    groups = [budgets[i::n] for i in range(n)]
    return [g for g in groups if g]

# One job per (seed, variant, budget-shard). `main.run` derives the run name,
# so ask it for the same name here to know which output files a job writes.
jobs = []
skipped = []
merges = []
for seed in SEEDS:
    for overrides, sampler_cfg in zip(VARIANTS, sampler_cfgs):
        name = RUN_NAME or main._default_run_name(SAMPLER, sampler_cfg)
        if seed != config.get("random_seed", 42):
            name = f"{name}_s{seed}"
        # Resume: a finished job wrote its merged results file. Re-running the
        # notebook after a session timeout then costs nothing for what is done.
        if (SAVE_DIR / f"{name}_results.pt").is_file():
            skipped.append(name)
            continue

        base_kwargs = dict(
            data_path=str(data_path),
            sampler_name=SAMPLER,
            num_classes=dataset_info["num_classes"],
            data_descriptions=dataset_info.get("descriptions", {}),
            prompt_templates=config.get("prompt_templates", []),
            sampler_cfg=sampler_cfg,
            probe_epochs=training_cfg["probe_epochs"],
            probe_lr=training_cfg["probe_lr"],
            random_seed=seed,
            save_dir=str(SAVE_DIR),
            verbose=True,
            model_cfg=config.get("models", {}),
            feature_cache_dir=FEATURE_DIR,
            run_name=name,
            device_string="cuda:0",
        )

        if shard_budgets:
            shards = budget_shards(BUDGETS, workers)
            tags = [f"shard{i}" for i in range(len(shards))]
            for tag, budgets in zip(tags, shards):
                jobs.append((f"{name}:{tag}", dict(
                    base_kwargs, cumulative_budget=budgets, shard_tag=tag,
                )))
            merges.append((name, tags))
        else:
            jobs.append((name, dict(base_kwargs, cumulative_budget=BUDGETS)))

if skipped:
    print(f"already finished, skipping {len(skipped)}: {', '.join(skipped)}")
print(f"to run: {len(jobs)} job(s) over {workers} GPU(s)")
for label, kwargs in jobs:
    print(f"   {label:40} budgets={kwargs['cumulative_budget']}")

started = time.time()
results = run_variants_parallel(jobs, main.run_on_worker, num_workers=workers)

print("=" * 70)
for result in results:
    status = "ok" if result["ok"] else "FAILED"
    print(f"{result['label']:40} {status:8} {format_duration(result['seconds'])}")
failed = [r["label"] for r in results if not r["ok"]]
print(f"total {format_duration(time.time() - started)} | "
      f"{len(results) - len(failed)}/{len(results)} succeeded")
assert not failed, f"variants failed: {failed}"

# Fold the shards back into the one `<run>_results.pt` an unsharded run would
# have written, so nothing downstream needs to know this run was split.
for name, tags in merges:
    linear = main.merge_budget_shards(str(SAVE_DIR), name, tags)
    print(f"[merge] {name}: {len(linear)} budgets -> {name}_results.pt")

In [ ]:
# Package the results as ONE zip at the top of /kaggle/working, then delete the
# loose files -- the same shape the two extraction notebooks use, and for the
# same reason.
#
# Kaggle's Output tab lists what is left in /kaggle/working when the session
# ends, and in a "Save & Run All" session that is the ONLY way to get a file
# out: there is no terminal and no kaggle CLI, so printing `kaggle datasets
# create` commands is advice nobody in that session can follow. The zip goes to
# the top level where it is easy to find, and the loose checkpoints are removed
# once it exists: keeping both means downloading everything twice, and a
# session over the ~20 GB Output quota shows NOTHING at all, including the
# files that were fine.
import shutil

from utils import results_archive_stem

SOURCE = Path(OUTPUT_DIR)
WORKING = Path("/kaggle/working")
assert SOURCE.is_dir(), f"nothing to archive at {SOURCE}"
# make_archive must not write inside the directory being archived, or it packs
# a partial copy of itself. OUTPUT_DIR is a subdirectory of WORKING, so writing
# to WORKING is safe -- assert it rather than assume it.
assert SOURCE.resolve() != WORKING.resolve(), (
    "OUTPUT_DIR must be a subdirectory of /kaggle/working, not /kaggle/working itself"
)

# dataset + sampler + seed are exactly the axes that make two result sets
# non-interchangeable, so they are the name.
STEM = results_archive_stem(DATASET, SAMPLER, SEEDS)
ARCHIVE = WORKING / STEM
shutil.make_archive(str(ARCHIVE), "zip", root_dir=SOURCE)
size_mb = ARCHIVE.with_suffix(".zip").stat().st_size / 1e6

# The zip root holds `<dataset>/`, which is the layout
# evaluate_al_sampler.ipynb expects after Kaggle extracts an uploaded dataset.
print(f"{ARCHIVE.name}.zip  ({size_mb:.1f} MB) contains:")
for path in sorted(SOURCE.rglob("*")):
    if path.is_file():
        print(f"    {path.relative_to(SOURCE)}  ({path.stat().st_size / 1e6:.2f} MB)")

# The zip is written and its size is known, so the originals are redundant.
shutil.rmtree(SOURCE, ignore_errors=True)

remaining = sorted(p for p in WORKING.iterdir() if p.name != "codapath")
total_mb = sum(
    f.stat().st_size for p in remaining for f in ([p] if p.is_file() else p.rglob("*"))
    if f.is_file()
) / 1e6
print(f"\n/kaggle/working now holds {total_mb:.1f} MB (Output quota ~20 GB):")
for path in remaining:
    print(f"    {path.name}{'/' if path.is_dir() else ''}")

print(f"""
NEXT STEPS (no terminal needed)
  1. Output tab (right panel) -> download {ARCHIVE.name}.zip
  2. kaggle.com/datasets -> New Dataset -> upload that zip
     Kaggle extracts it into a directory named after the zip, so the
     checkpoints end up one level down. That is expected.
  3. In evaluate_al_sampler.ipynb: Add Data -> your new dataset, then point
     CHECKPOINT_ROOT at it to rebuild the table and fit PALM.""")